# Assignment 1 — QANet

**COMP5329 / Deep Learning — University of Sydney, Semester 1 2026**

Run each section in order. Sections 0–1 are one-time setup steps; Sections 2–4 are the main training and evaluation pipeline.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# # Adjust this path if your repo is stored elsewhere in Drive.
# PROJECT_ROOT = "/content/drive/MyDrive/Assignment1_2026"

In [4]:
# # Install Python dependencies (run once per session)
# !pip install -r {PROJECT_ROOT}/requirements.txt -q
# !python -m spacy download en

---
## Section 0 — Environment Setup

Mount Google Drive and install dependencies.

In [5]:
import sys, os
# local root
from pathlib import Path
PROJECT_ROOT = str(Path.cwd().resolve())

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

Working directory: E:\Project_usyd\COMP5329_Assignment1


---
## Section 1 — Download Data *(delete before submitting)*

Downloads the pre-built mini dataset (sampled SQuAD v1.1 train + full dev set,
with GloVe vectors filtered to the mini vocabulary) from GitHub Releases into `_data/`.

> **One-time step.** Once `_data/` exists on your Drive, delete this section before submission.

In [7]:
from Tools.download import download_mini

download_mini(data_dir="_data")

Step 1 / 2  —  Mini dataset (SQuAD + GloVe)
  [skip] Mini dataset already present in _data/.

Step 2 / 2  —  spaCy language model
⚠ As of spaCy v3.0, shortcuts like 'en' are deprecated. Please use the
full pipeline package name 'en_core_web_sm' instead.
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------- ----------------- 7.3/12.8 MB 41.4 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 36.6 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')

Mini dataset download complete.


---
## Section 2 — Preprocess Data *(delete before submitting)*

Tokenises the SQuAD JSON files, builds word/char vocabularies from GloVe, and writes padded index tensors to `_data/`.

> **One-time step.** Once `_data/*.npz` exists on your Drive, delete this section before submission. Re-run only if you change `para_limit`, `ques_limit`, or other shape parameters.

In [1]:
from Tools.preproc import preprocess

preprocess(
    train_file="_data/squad/train-mini.json",
    dev_file="_data/squad/dev-v1.1.json",
    glove_word_file="_data/glove/glove.mini.txt",
    target_dir="_data",
    para_limit=400,
    ques_limit=50,
)

Generating train examples…


100%|██████████| 150/150 [00:04<00:00, 33.86it/s]


  30293 questions in total
Generating dev examples…


100%|██████████| 48/48 [00:02<00:00, 22.14it/s]


  10570 questions in total
Generating word embedding…


...

100%|██████████| 30293/30293 [00:08<00:00, 3582.36it/s]


  Built 30169 / 30293 instances
Processing dev examples…


100%|██████████| 10570/10570 [00:03<00:00, 3054.17it/s]


  Built 10465 / 10570 instances
Saving word embedding…
Saving char embedding…
Saving train eval…
Saving dev eval…
Saving word dictionary…
Saving char dictionary…
Saving dev meta…

Preprocessing complete.
  Outputs → _data/


{'train_record_file': '_data\\train.npz',
 'dev_record_file': '_data\\dev.npz',
 'word_emb_file': '_data\\word_emb.json',
 'char_emb_file': '_data\\char_emb.json',
 'train_eval_file': '_data\\train_eval.json',
 'dev_eval_file': '_data\\dev_eval.json',
 'word2idx_file': '_data\\word2idx.json',
 'char2idx_file': '_data\\char2idx.json',
 'dev_meta_file': '_data\\dev_meta.json'}

---
## Section 3 — Train

Trains QANet on SQuAD v1.1 and saves the best checkpoint to `_model/model.pt`.

In [3]:
from TrainTools.train import train

results = train(
    # ── data paths (must match preprocess outputs) ──────────────────────
    train_npz       = "_data/train.npz",
    dev_npz         = "_data/dev.npz",
    word_emb_json   = "_data/word_emb.json",
    char_emb_json   = "_data/char_emb.json",
    train_eval_json = "_data/train_eval.json",
    dev_eval_json   = "_data/dev_eval.json",
    save_dir        = "_model",
    log_dir         = "_log",

    # ── training loop ────────────────────────────────────────────────────
    num_steps  = 1000,
    batch_size = 8,
    seed       = 42,

    # ── vanilla recipe: SGD, no scheduler, NLL loss ───────────────────────
    optimizer_name = "sgd",
    scheduler_name = "lambda",
    loss_name      = "qa_nll",
)

print(f"Best F1: {results['best_f1']:.4f}  |  Best EM: {results['best_em']:.4f}")

100%|██████████| 200/200 [18:40<00:00,  5.60s/it]


STEP      200  loss 1837.581248



100%|██████████| 150/150 [02:29<00:00,  1.00it/s]


VALID(train) loss 33.581099  F1 7.250501  EM 0.000000



100%|██████████| 150/150 [02:19<00:00,  1.07it/s]


...

STEP     1000  loss 213.217089



100%|██████████| 150/150 [02:23<00:00,  1.04it/s]


VALID(train) loss 16.617065  F1 7.275904  EM 0.333333



100%|██████████| 150/150 [02:28<00:00,  1.01it/s]


TEST        loss 16.928530  F1 5.185675  EM 0.000000

Learning rate: [0.001]
Training finished.  Best F1: 6.1272  Best EM: 0.1667
Best F1: 6.1272  |  Best EM: 0.1667


---
## Section 4 — Evaluate

Loads the saved checkpoint and runs inference on the full dev set.

In [1]:
from EvaluateTools.evaluate import evaluate

metrics = evaluate(
    dev_npz       = "_data/dev.npz",
    word_emb_json = "_data/word_emb.json",
    char_emb_json = "_data/char_emb.json",
    dev_eval_json = "_data/dev_eval.json",
    save_dir      = "_model",
    log_dir       = "_log",
    ckpt_name     = "model.pt",
)

print(f"F1: {metrics['f1']:.4f}  |  EM: {metrics['exact_match']:.4f}  |  Loss: {metrics['loss']:.6f}")

100%|██████████| 1309/1309 [19:39<00:00,  1.11it/s]


TEST  loss 17.511563  F1 7.657735  EM 0.143335
F1: 7.6577  |  EM: 0.1433  |  Loss: 17.511563
